# Praxis · Reproduce our validation — *Downside Risk Reduction Using Regime-Switching Signals: A Statistical Jump Model Approach* (#363490)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finique/praxis-reproductions/blob/main/363490-jump-model-regime-timing.ipynb)

**Paper:** *Downside Risk Reduction Using Regime-Switching Signals: A Statistical Jump Model Approach* (2024), by Yizhan Shu, Chenyu Yu & John M. Mulvey (Princeton ORFE) — [arXiv 2402.05272](https://arxiv.org/abs/2402.05272). The authors publish their model code as [`jumpmodels`](https://github.com/Yizhan-Oliver-Shu/jump-models); this notebook uses it for the fit.

This notebook **reproduces our exact verdict** on the paper's headline claim — that a two-state statistical jump model, read online on an equity index's own returns, times the market well enough for a 100 %-index / 100 %-bills switch to beat buy-and-hold on the Sharpe ratio (S&P 500: **0.68 vs 0.48**, 1990–2023, one-day delay, 10 bp costs). From **public data only**, frozen in this repository (`363490-data/`): the S&P 500 total return (the real total-return index from 1988, Shiller's dividends accrued onto the price index before that — 0.9996 correlation on the overlap), the DAX (a total-return index) and the Nikkei 225 (price index plus a documented dividend-yield proxy), each with its own 3-month bill from FRED. It:
1. builds the paper's **three smoothed features** (downside deviation, two Sortino ratios) from daily excess returns,
2. refits the **two-state jump model** every six months on 3,000 days with the authors' package, and infers the regime **online** every day by dynamic programme over the trailing 3,000 days,
3. re-picks the **jump penalty λ every month** by the paper's 8-year Sharpe-maximising cross-validation, runs the delayed, cost-adjusted **0/1 strategy**, and prints **claimed → measured**,
4. adds the **two baselines the paper never shows** — a constant mix at the strategy's own average exposure and a 200-day moving average — and a **paired block bootstrap** of the Sharpe difference.

**Our published verdict: FAILS — the risk reduction is real, the Sharpe enhancement is not.** On the S&P 500 the strategy earns exactly the buy-and-hold Sharpe (0.495 vs 0.496); a constant 72 % mix ties it and the 200-day moving average beats it. The DAX is worse than buy-and-hold on its honest 2008–2023 window; only the Nikkei keeps the direction, without significance.

Set `INDEX` in the run cell to `"SP500"` (default), `"DAX"` or `"N225"`. The nine λ × ~90 refits take about 10–15 minutes in Colab. Same data and logic as our engine — this lands on the report's numbers to the decimal. The full C0–C7 gate suite, the fragility rungs and the reduced HMM benchmark live in the report: https://market-graphs.com/praxis/papers/363490


In [ ]:
%pip install -q "jumpmodels==0.1.1" "scikit-learn==1.8.0" "pandas==2.2.3" "numpy==2.1.2"

In [ ]:
"""Praxis · reproduce our validation of paper 363490 "Downside Risk Reduction
Using Regime-Switching Signals: A Statistical Jump Model Approach" (Shu, Yu &
Mulvey, Princeton, 2024, arXiv 2402.05272).

Self-contained. PUBLIC data only, frozen in this repository (363490-data/):
daily total-return index + 3-month bill series for the S&P 500 (^GSPC +
Shiller dividends to 1987, the real ^SP500TR from 1988), the DAX (^GDAXI, a
total-return index) and the Nikkei 225 (^N225 + a documented dividend-yield
proxy), with bills from FRED (DTB3; OECD/IMF 3-month rates for Germany and
Japan). Model fitting uses the AUTHORS' own package (`jumpmodels`).

Pipeline (paper §3.4): three EWM features of the index's own excess returns
(downside deviation hl=10; Sortino hl=20, 60) → two-state jump model refit
every six months on 3,000 days (10 k-means++ inits, bull = higher training
cumulative excess return) → daily online inference by dynamic programme over
the trailing 3,000 days with centroids fixed → jump penalty λ re-selected
monthly by 8-year time-series cross-validation on the delayed, cost-adjusted
strategy Sharpe → 0/1 strategy: 100 % index in bull, 100 % bills in bear,
one-day delay, 10 bp one-way costs. Prints claimed → measured (paper Table 4)
plus the two baselines the paper never shows (a constant mix at the strategy's
own average exposure; a 200-day moving-average rule) and a paired block
bootstrap of the Sharpe difference.
"""
import io
import time
import urllib.request

import numpy as np
import pandas as pd
from jumpmodels.jump import JumpModel

INDEX = "SP500"                 # "SP500" · "DAX" · "N225"
DATA_URL = "https://raw.githubusercontent.com/finique/praxis-reproductions/main/363490-data/"
GRID = (5.0, 10.0, 15.0, 25.0, 35.0, 50.0, 70.0, 100.0, 150.0)
TRAIN_LEN = LOOKBACK = 3000
COST_BPS, DELAY, CV_YEARS, PPY = 10.0, 1, 8, 252
PAPER_START, PAPER_END, POST_START, SAMPLE_END = "1990-01-01", "2023-12-31", "2024-01-01", "2026-08-31"
CLAIMED = {"SP500": {"bh": (10.2, 18.2, 0.48, -55.2), "jm": (11.2, 13.1, 0.68, -26.6, 44, 80)},
           "DAX": {"bh": (6.8, 22.1, 0.30, -72.7), "jm": (8.6, 16.4, 0.44, -39.4, 170, 84)},
           "N225": {"bh": (0.8, 23.4, 0.12, -79.1), "jm": (4.7, 17.1, 0.31, -45.3, 72, 75)}}


# ------------------------------------------------------------------- data ---
def load(index: str) -> pd.DataFrame:
    with urllib.request.urlopen(DATA_URL + f"{index}.csv", timeout=60) as r:
        df = pd.read_csv(io.BytesIO(r.read()), parse_dates=["date"],
                         float_precision="round_trip").set_index("date")
    return df.loc[:SAMPLE_END]        # columns: ret (daily total return), rf (daily bill, decimal)


# --------------------------------------------------------------- features ---
def ewm_dd(r, hl):
    return np.sqrt(pd.Series(np.minimum(r, 0.0), index=r.index).pow(2).ewm(halflife=hl).mean())


def ewm_features(excess, burn=0):
    f = pd.DataFrame({"dd_10": ewm_dd(excess, 10),
                      "sortino_20": excess.ewm(halflife=20).mean() / ewm_dd(excess, 20),
                      "sortino_60": excess.ewm(halflife=60).mean() / ewm_dd(excess, 60)})
    return f.replace([np.inf, -np.inf], np.nan).iloc[burn:].dropna()


# --------------------------------------------------- fit + online inference ---
def refit_schedule(index):
    s = pd.Series(index, index=index)
    ends = s.groupby(index.to_period("M")).last()
    pos = {d: i for i, d in enumerate(index)}
    return [d for d in ends if d.month in (6, 12) and pos[d] + 1 >= TRAIN_LEN]


def online_states(dl, lam, ends):
    starts = np.maximum(ends - LOOKBACK + 1, 0)
    D = np.zeros(len(ends))
    for k in range(LOOKBACK):
        pos = starts + k
        m = pos <= ends
        if not m.any():
            break
        D[m] = np.clip(D[m], -lam, lam) + dl[pos[m]]
    return (D > 0).astype(int)          # V(bull) − V(bear) > 0 → bear


def run_pipeline(feats, excess, lam):
    X, r, idx = feats.values, excess.reindex(feats.index).values, feats.index
    pos = {d: i for i, d in enumerate(idx)}
    refits = refit_schedule(idx)
    states = np.full(len(idx), np.nan)
    for p, d in enumerate(refits):
        i = pos[d]
        Xtr = X[i - TRAIN_LEN + 1: i + 1]
        mean, scale = Xtr.mean(0), Xtr.std(0)
        jm = JumpModel(n_components=2, jump_penalty=lam, cont=False, random_state=0, n_init=10)
        jm.fit((Xtr - mean) / scale, r[i - TRAIN_LEN + 1: i + 1], sort_by="cumret")
        c = np.asarray(jm.centers_, dtype=float)
        j0 = i + 1
        j1 = pos[refits[p + 1]] if p + 1 < len(refits) else len(idx) - 1
        if j0 > j1:
            continue
        lo = max(0, j0 - LOOKBACK + 1)
        Z = (X[lo: j1 + 1] - mean) / scale
        l0 = 0.5 * ((Z - c[0]) ** 2).sum(1)
        l1 = 0.5 * ((Z - c[1]) ** 2).sum(1)
        dl = np.where(np.isnan(l0), np.inf, l0) - np.where(np.isnan(l1), np.inf, l1)
        states[j0: j1 + 1] = online_states(dl, lam, np.arange(j0, j1 + 1) - lo)
    return pd.Series(states, index=idx)


# --------------------------------------------------------------- strategy ---
def strategy(w, ret, rf, cost_bps=COST_BPS):
    w = w.dropna()
    R, f = ret.reindex(w.index), rf.reindex(w.index)
    dw = w.diff().abs().fillna(0.0)
    return pd.DataFrame({"w": w, "r": w * R + (1 - w) * f - cost_bps / 1e4 * dw, "dw": dw})


def zero_one(states, ret, rf, delay=DELAY, cost_bps=COST_BPS):
    return strategy((1.0 - states).shift(1 + delay), ret, rf, cost_bps)


def select_lambda_monthly(net, rf):
    first = net.dropna().index[0]
    rows = []
    s = pd.Series(net.dropna().index, index=net.dropna().index)
    for t in s.groupby(s.index.to_period("M")).last():
        lo = t - pd.DateOffset(years=CV_YEARS)
        if lo < first:
            continue
        e = net.loc[net.index > lo].loc[:t]
        e = e.sub(rf.reindex(e.index), axis=0)
        sr = e.mean() / e.std(ddof=1) * np.sqrt(PPY)
        rows.append((t, float(sr.idxmax())))
    return pd.Series(dict(rows))


def compose(S, lam_sel, ret, rf, delay=DELAY, cost_bps=COST_BPS):
    idx = ret.index
    lam_u = lam_sel.reindex(idx).shift(1 + delay).ffill()
    Sd = S.reindex(idx).shift(1 + delay)
    cols = np.array([float(c) for c in Sd.columns])
    st = np.full(len(idx), np.nan)
    ok = lam_u.notna().values
    st[ok] = Sd.values[ok, np.searchsorted(cols, lam_u.values[ok])]
    w = pd.Series(1.0 - st, index=idx)
    w[~ok] = np.nan
    return strategy(w, ret, rf, cost_bps)


def sma_states(tr, n=200):
    ma = tr.rolling(n).mean()
    s = (tr < ma).astype(float)
    s[ma.isna()] = np.nan
    return s


def constant_mix(ret, rf, e, cost_bps=COST_BPS):
    w_pre = (e * (1 + ret)) / (e * (1 + ret) + (1 - e) * (1 + rf))
    dw = (w_pre - e).abs().fillna(0.0)
    return pd.DataFrame({"w": e, "r": e * ret + (1 - e) * rf - cost_bps / 1e4 * dw})


# ---------------------------------------------------------------- metrics ---
def mdd(r):
    w = (1 + r).cumprod()
    return float((w / w.cummax() - 1).min())


def stats(r, rf, w=None):
    r = r.dropna()
    e = r - rf.reindex(r.index)
    yrs = len(r) / PPY
    out = dict(ret=((1 + r).prod() ** (1 / yrs) - 1) * 100, vol=r.std(ddof=1) * np.sqrt(PPY) * 100,
               sharpe=e.mean() / e.std(ddof=1) * np.sqrt(PPY), mdd=mdd(r) * 100)
    if w is not None:
        w = w.reindex(r.index)
        out["turnover"] = w.diff().abs().fillna(0).sum() / 2 / yrs * 100
        out["leverage"] = w.mean() * 100
    else:
        out["turnover"], out["leverage"] = 0.0, 100.0
    return out


def block_bootstrap(r_a, r_b, rf, block=63, n_boot=2000, seed=0):
    j = pd.concat([r_a.rename("a"), r_b.rename("b"), rf.reindex(r_a.index).rename("rf")], axis=1).dropna()
    A = j.values
    T = len(A)
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(T / block))
    d = np.empty(n_boot)
    for b in range(n_boot):
        pos = (rng.integers(0, T, size=nb)[:, None] + np.arange(block)[None, :]).ravel()[:T] % T
        s = A[pos]
        xa, xb = s[:, 0] - s[:, 2], s[:, 1] - s[:, 2]
        d[b] = (xa.mean() / xa.std(ddof=1) - xb.mean() / xb.std(ddof=1)) * np.sqrt(PPY)
    return float((d <= 0).mean()), [float(np.quantile(d, .025)), float(np.quantile(d, .975))]


In [ ]:
# --------------------------------------------------------------------- run ---
def main(index=INDEX):
    t0 = time.time()
    df = load(index)
    ret, rf = df["ret"], df["rf"]
    excess = ret - rf
    tr = 100 * (1 + ret).cumprod()
    feats = ewm_features(excess)
    print(f"{index}: {ret.index[0].date()} → {ret.index[-1].date()}, {len(ret)} days; "
          f"fitting {len(GRID)} λ × {len(refit_schedule(feats.index))} refits …", flush=True)
    S = {}
    for lam in GRID:
        S[lam] = run_pipeline(feats, excess, lam)
        print(f"  λ={lam:>5g} done ({time.time() - t0:.0f}s)", flush=True)
    S = pd.DataFrame(S).reindex(ret.index)
    fixed = {lam: zero_one(S[lam], ret, rf) for lam in GRID}
    sel = select_lambda_monthly(pd.DataFrame({lam: f["r"] for lam, f in fixed.items()}), rf)
    oos_all = compose(S, sel, ret, rf)
    start = max(pd.Timestamp(PAPER_START), sel.index[0] + pd.DateOffset(days=1))
    for label, (a, b) in {"paper window": (start, PAPER_END), "post-publication": (POST_START, SAMPLE_END)}.items():
        o = oos_all.loc[a:b]
        if len(o) < 250:
            continue
        bh = ret.reindex(o.index)
        cm = constant_mix(bh, rf.reindex(o.index), o["w"].mean())
        sma = zero_one(sma_states(tr), ret, rf).loc[a:b]
        rows = {"JM 0/1 (λ by CV)": stats(o["r"], rf, o["w"]), "Buy & hold": stats(bh, rf),
                f"Constant {o['w'].mean() * 100:.0f}% mix": stats(cm["r"], rf, cm["w"]),
                "200-day MA rule": stats(sma["r"], rf, sma["w"])}
        print(f"\n=== {index} · {label} {o.index[0].date()} → {o.index[-1].date()} ===")
        print(f"{'':22}{'CAGR%':>8}{'vol%':>7}{'Sharpe':>8}{'MDD%':>8}{'turn%':>7}{'exp%':>6}")
        for k, s in rows.items():
            print(f"{k:22}{s['ret']:8.2f}{s['vol']:7.1f}{s['sharpe']:8.3f}{s['mdd']:8.1f}{s['turnover']:7.1f}{s['leverage']:6.0f}")
        if label == "paper window":
            c = CLAIMED[index]
            print(f"{'claimed JM':22}{c['jm'][0]:8.2f}{c['jm'][1]:7.1f}{c['jm'][2]:8.3f}{c['jm'][3]:8.1f}{c['jm'][4]:7.1f}{c['jm'][5]:6.0f}")
            print(f"{'claimed B&H':22}{c['bh'][0]:8.2f}{c['bh'][1]:7.1f}{c['bh'][2]:8.3f}{c['bh'][3]:8.1f}")
        p_bh, ci = block_bootstrap(o["r"], bh, rf)
        p_cm, _ = block_bootstrap(o["r"], cm["r"], rf)
        p_sma, _ = block_bootstrap(o["r"], sma["r"], rf)
        print(f"paired block-bootstrap p(ΔSharpe ≤ 0): vs B&H {p_bh:.3f} (95% CI {ci[0]:+.2f}…{ci[1]:+.2f}) · "
              f"vs constant mix {p_cm:.3f} · vs 200-day MA {p_sma:.3f}")
        ch = (sel.loc[a:b].diff().abs() > 0).mean()
        print(f"λ re-selection changed the penalty in {ch * 100:.0f}% of months")
    return oos_all, sel


oos, lam_sel = main(INDEX)


## What you just reproduced (S&P 500, 1990-02 → 2023-12)
- **Jump-model 0/1 strategy: Sharpe 0.495, CAGR 8.33 %, max drawdown −34.4 %** against buy-and-hold **0.496, 10.47 %, −55.3 %** — the paper claimed 0.68 / 11.2 % / −26.6 % against 0.48 / 10.2 % / −55.2 %. The buy-and-hold row lands on the paper's; the strategy row does not.
- **The risk reduction is real** (volatility 12.5 % vs 18.2 %, drawdown cut by 21 points) — and it is what any reduced exposure delivers: the **constant 72 % mix scores 0.493**, the **200-day moving average 0.523** with a −23 % drawdown.
- **The paired block bootstrap** puts the Sharpe difference against buy-and-hold at −0.001 with a 95 % interval of −0.27 … +0.28 (p = 0.52); against the constant mix p = 0.51; against the moving average p = 0.56.
- The penalty re-selection changed λ in **7.8 % of months** and sat at the grid's smoothest value (150) 39 % of the time.

Run `INDEX = "DAX"` (out-of-sample 2008-02 → 2023-12 on public data: 0.151 vs 0.330 — worse than the index) and `"N225"` (1990-08 → 2023-12: 0.285 vs 0.164, p = 0.23 — the one index that keeps the direction). The post-publication window (2024-01 → 2026-08) prints too: the strategy trails buy-and-hold on all three.

**Verdict: FAILS — a genuine drawdown reducer that is not a Sharpe enhancer.** The crash years are spent in bills and won; the rebound years are spent in bills and lost. The C0–C7 ladder, the fragility rungs (grid to 500 → 0.525; the authors' example-code clipping → 0.568; best fixed λ → 0.591), the reduced HMM benchmark (0.58, where the paper put it) and the look-ahead / shuffle / truncation traps are in the report: https://market-graphs.com/praxis/papers/363490
